In [11]:
import pandas as pd
import re
df = pd.read_csv('offers.csv',encoding='utf-8',sep=';')
clean_df = pd.DataFrame()
def clean_price(price_str):
    if pd.isna(price_str): return None
    digits = re.sub(r'\D', '', str(price_str))
    return int(digits) if digits else None
clean_df['price'] = df['Цена'].apply(clean_price)
def clean_area(area_str):
    if pd.isna(area_str): return None
    first_part = str(area_str).split('/')[0]
    try:
        return float(first_part)
    except:
        return None
clean_df['square_meters'] = df['Площадь, м2'].apply(clean_area)
def get_minutes_to_subway(metro_str):
    if pd.isna(metro_str): return None
    match = re.search(r'(\d+)\s*мин', str(metro_str))
    if match:
        return int(match.group(1))
    return None
clean_df['minutes_to_subway'] = df['Метро'].apply(get_minutes_to_subway)
clean_df['parking'] = df['Парковка'].fillna('Не указано')
clean_df['has_parking'] = clean_df['parking'].isin(['подземная', 'наземная']).astype(int)
clean_df = clean_df.dropna(subset=['price', 'square_meters', 'minutes_to_subway'])

,price,square_meters,minutes_to_subway,parking,has_parking
0,74250000,55.00,14,Не указано,0
1,36000000,43.70,5,Не указано,0
2,56700000,62.00,10,Не указано,0
3,31890000,45.00,24,подземная,1
4,18300000,33.00,5,Не указано,0
...,...,...,...,...,...
195,13800000,38.60,4,Не указано,0
196,32120000,54.30,13,Не указано,0
197,16900000,32.10,20,подземная,1
198,27250000,43.30,7,подземная,1


In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
x = clean_df[['square_meters','minutes_to_subway','has_parking']]
y = clean_df['price']
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state = 67)
model = RandomForestRegressor(n_estimators = 100, max_depth = 3,random_state = 67)
model.fit(x_train,y_train)
y_pred = model.predict(x_test)

print(f"Средняя ошибка предсказания (MAE): {mean_absolute_error(y_test, y_pred):,.2f}")
print(f"Качество модели (R² score): {r2_score(y_test, y_pred):.2f}")

Средняя ошибка предсказания (MAE): 9,775,590.20
Качество модели (R² score): 0.19


In [20]:
from sklearn.model_selection import cross_val_score
import numpy as np

scores = cross_val_score(model, x, y, cv=5, scoring='r2')
print(f"Средний честный R²: {np.mean(scores):.2f}")
print(f"Разброс R² по складкам: {scores}")

Средний честный R²: 0.36
Разброс R² по складкам: [0.50296274 0.18059994 0.31576394 0.27083002 0.53769406]
